# UMAP Feature Analysis

This notebook creates UMAP visualizations of latent and manual features, colored by attention and age score.
It also explores feature importance based on Integrated Gradients (IG).


In [ ]:
import pandas as pd
import numpy as np
import umap
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler

# Paths
ATTENTION_PATH = "../outputs/logs/baseline_v2_attention.csv"
LABELS_PATH = "../data/processed/labels.csv"
MANUAL_FEATS_PATH = "../data/processed/features_manual.csv"
LATENT_FEATS_PATH = "../data/processed/features_latent.csv"
IMPORTANCE_PATH = "../outputs/logs/baseline_v2_importance_normalised.csv"

# Load data
df_att = pd.read_csv(ATTENTION_PATH)
df_labels = pd.read_csv(LABELS_PATH)
df_manual = pd.read_csv(MANUAL_FEATS_PATH)
df_latent = pd.read_csv(LATENT_FEATS_PATH)
df_imp = pd.read_csv(IMPORTANCE_PATH)

# Ensure indices match (they seem to be aligned based on exploration)
# But we'll use well_name and roi for safety if needed. 
# Here we assume rows are aligned across features and labels/attention files.
df_full = pd.concat([df_labels[['well_name', 'roi', 's']], 
                     df_att[['mean_attention']], 
                     df_manual, 
                     df_latent], axis=1)

manual_cols = df_manual.columns.tolist()
latent_cols = df_latent.columns.tolist()
all_feat_cols = manual_cols + latent_cols

print(f"Total cells: {len(df_full)}")
print(f"Manual features: {len(manual_cols)}")
print(f"Latent features: {len(latent_cols)}")


In [ ]:
df_manual

## 1. UMAP Visualizations

We will create UMAPs for:
1. Latent features individually
2. Manual features individually
3. Combined features


In [ ]:
def get_umap_embedding(df, feat_cols, scale_weights=None):
    # Scale features
    scaler = StandardScaler()
    X = scaler.fit_transform(df[feat_cols])
    
    if scale_weights is not None:
        X = X * scale_weights
        
    # Run UMAP
    reducer = umap.UMAP(random_state=42, n_neighbors=15, n_components=2, min_dist=0.1)
    embedding = reducer.fit_transform(X)
    return embedding

def add_umap_trace(fig, embedding, df, color_col, row, col, marker_size=4, marker_opacity=0.8):
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]

    temp_fig = px.scatter(
        df_plot, x='UMAP1', y='UMAP2',
        color=color_col,
        hover_data=['well_name', 'roi'],
        color_continuous_scale='Viridis',
        render_mode='svg'
    )

    trace = temp_fig.data[0]
    trace.marker.coloraxis = f'coloraxis{col}'
    trace.marker.size = marker_size
    trace.marker.opacity = marker_opacity
    fig.add_trace(trace, row=row, col=col)

    fig.update_xaxes(title_text="UMAP1", row=row, col=col)
    fig.update_yaxes(title_text="UMAP2", row=row, col=col, scaleanchor=f"x{col}", scaleratio=1)


def create_umap_grid(df, feat_cols, title_prefix, scale_weights=None, marker_size=4, marker_opacity=0.8):
    embedding = get_umap_embedding(df, feat_cols, scale_weights)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f"{title_prefix} - Attention", f"{title_prefix} - Age Score"),
        horizontal_spacing=0.15
    )

    add_umap_trace(fig, embedding, df, 'mean_attention', 1, 1, marker_size, marker_opacity)
    add_umap_trace(fig, embedding, df, 's', 1, 2, marker_size, marker_opacity)

    fig.update_layout(
        height=800, width=1600,
        title_text=title_prefix,
        coloraxis1=dict(colorscale='Viridis', colorbar=dict(title='Attention', x=0.42, thickness=15)),
        coloraxis2=dict(colorscale='Viridis', colorbar=dict(title='Age Score', x=1.0, thickness=15)),
        showlegend=False
    )
    fig.show()

# 1.1 Latent Features
create_umap_grid(df_full, latent_cols, "Latent Features")

# 1.2 Manual Features
create_umap_grid(df_full, manual_cols, "Manual Features")

# 1.3 Combined Features
create_umap_grid(df_full, all_feat_cols, "Combined Features")


## 2. Importance-Weighted UMAPs

Next, we use the top X features by weighted IG importance.


In [ ]:
# Top X features by global_mean_abs_ig
X_TOP = 6
top_features = df_imp.sort_values('global_mean_abs_ig', ascending=False).head(X_TOP)['feature'].tolist()

print(f"Top {X_TOP} features by IG importance:")
print(top_features)

# 2.1 UMAP with Top X features
create_umap_grid(df_full, top_features, f"Top {X_TOP} IG Features")


## 3. Scaled Feature UMAP

Scale each feature based on its IG importance before running UMAP.


In [ ]:
# Create importance weight vector
imp_map = df_imp.set_index('feature')['global_mean_abs_ig'].to_dict()
weights = np.array([imp_map.get(c, 0.0) for c in all_feat_cols])

create_umap_grid(df_full, all_feat_cols, "All Features Scaled", scale_weights=weights)


In [ ]:
 import lightgbm as lgb
import shap
from sklearn.model_selection import train_test_split

# ── Prepare data ──────────────────────────────────────────────────────────
X = df_full[all_feat_cols].copy()
y = df_full['mean_attention'].copy()

# Drop rows with any NaNs across features/target (SHAP/LightGBM need clean input)
valid_mask = X.notna().all(axis=1) & y.notna()
X, y = X[valid_mask], y[valid_mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Fit surrogate model ──────────────────────────────────────────────────
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
)

train_r2 = model.score(X_train, y_train)
test_r2 = model.score(X_test, y_test)
print(f"Surrogate fit quality — train R²: {train_r2:.4f} | test R²: {test_r2:.4f}")

# ── SHAP values ───────────────────────────────────────────────────────────
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# ── Global ranked feature importance (mean |SHAP|) ──────────────────────
shap_importance = pd.DataFrame({
    'feature': X_test.columns,
    'feature_type': ['manual' if c in manual_cols else 'latent' for c in X_test.columns],
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print("\nTop 15 features driving attention (mean |SHAP|):")
print(shap_importance.head(15).to_string(index=False))

# ── Plots ─────────────────────────────────────────────────────────────────
shap.summary_plot(shap_values, X_test, max_display=20, show=True)

# Dependence plots for top 4 features — reveals monotonic / threshold / U-shape behaviour
top_feats = shap_importance['feature'].head(4).tolist()
for feat in top_feats:
    shap.dependence_plot(feat, shap_values, X_test, show=True)